# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/dev-hashh/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 135 (delta 47), reused 84 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.90 MiB | 7.54 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule Idea

Goal: Prioritize content pages for refresh review.

Hypothesis 1 (Staleness):
Pages that have not been updated recently are more likely to show weaker performance and deserve review.

Hypothesis 2 (Visibility):
Pages with higher impressions represent larger business opportunities and should be prioritized when stale.

Signal Verdicts:
- Staleness (days_since_last_update): TBD
- Visibility (impressions_90d): TBD

Reason Code:
- stale_visible_page

Action Label:
- REFRESH

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


# Signal Check 1 — Staleness

In [ ]:
staleness = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_impressions=("impressions_90d", "mean"),
          avg_sessions=("sessions_90d", "mean")
      )
      .reset_index()
)

display(staleness)

,freshness_tier,n,avg_impressions,avg_sessions
0,0-30,20480,4199.614062,24.453662
1,181+,174,1172.448276,5.540230
2,31-90,175,6506.748571,28.171429
3,91-180,9171,7486.665140,66.000872


### Signal Audit 1: Staleness

Signal: freshness_tier

Observation:
Pages in the oldest bucket (181+ days since update) have substantially lower average impressions than other groups, suggesting stale content may require review. However, the relationship is not strictly monotonic because pages in the 91–180 day bucket have the highest average impressions.

Verdict: MIXED

Evidence:
- 0-30: 4,200 avg impressions (n=20,480)
- 31-90: 6,507 avg impressions (n=175)
- 91-180: 7,487 avg impressions (n=9,171)
- 181+: 1,172 avg impressions (n=174)

# Signal Check 2 — Visibility

In [ ]:
visibility = (
    df.groupby("impression_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_sessions=("sessions_90d", "mean")
      )
      .reset_index()
)

display(visibility)

,impression_tier,n,avg_ctr,avg_sessions
0,excellent,1078,0.312662,296.674397
1,good,7205,0.308314,76.889244
2,low,11248,0.937000,4.762180
3,moderate,10469,0.212453,17.636068


### Signal Audit 2: Visibility

Signal: impression_tier

Observation:
Pages with higher visibility consistently generate substantially more sessions. The relationship is strong and monotonic across all buckets.

Verdict: CONFIRMED

Evidence:
- Excellent: 296.7 avg sessions (n=1,078)
- Good: 76.9 avg sessions (n=7205)
- Moderate: 17.6 avg sessions (n=10,469)
- Low: 4.8 avg sessions (n=11,248)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
df["visibility_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)

df["staleness_score"] = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
)

# df["baseline_score"] = (
#     0.7 * df["visibility_score"] +
#     0.3 * df["staleness_score"]
# )

df["baseline_score"] = (
    df["visibility_score"] *
    df["staleness_score"]
)

conditions = [
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500),
    (df["impressions_90d"] >= 1000),
]

choices = [
    "stale_visible_page",
    "high_visibility_page"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="monitor"
)

df["action"] = np.where(
    df["reason_code"] == "stale_visible_page",
    "REFRESH",
    "MONITOR"
)

In [ ]:
# Save the csv output
import os

os.makedirs("/content/flyrank-ml-internship-starter/outputs", exist_ok=True)

df.to_csv(
    "/content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv",
    index=False
)

In [ ]:
df = pd.read_csv(
    f"/content/flyrank-ml-internship-starter/outputs/baseline_action_score.csv"
)

print("Rows: ", len(df))
print("Columns: ", len(df.columns))
print(df.head())

Rows:  30000
Columns:  50
             content_id          client_id  search_volume  competition  \
0  content_5fe46e04994d  client_4e07408562         1900.0         0.00   
1  content_2cb567c3c89b  client_6208ef0f77            0.0         0.00   
2  content_2dba2b1f9536  client_6208ef0f77            0.0         0.00   
3  content_aaef01a50def  client_19581e27de         4400.0         0.05   
4  content_8c19996aa890  client_4e07408562           70.0         0.01   

  competition_level   cpc     content_type    main_intent  word_count  \
0               LOW  0.00  keyword article  informational         NaN   
1               LOW  0.00  keyword article  informational      6183.0   
2               LOW  0.00  keyword article  informational      7676.0   
3               LOW  0.11  keyword article  informational         NaN   
4               LOW  0.00  keyword article  informational      2895.0   

   char_count  ... impression_tier position_tier  trend_direction  trend_pct  \
0         

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = df.sort_values(
    "baseline_score",
    ascending=False
).head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "impressions_90d",
        "days_since_last_update",
        "reason_code",
        "action"
    ]
]

,content_id,baseline_score,impressions_90d,days_since_last_update,reason_code,action
0,content_5fe46e04994d,0.278820,517715,104,high_visibility_page,MONITOR
2,content_2dba2b1f9536,0.238816,443434,104,high_visibility_page,MONITOR
6,content_2c2606c5d176,0.187095,347399,104,high_visibility_page,MONITOR
8,content_cb112fce36be,0.166905,309910,104,high_visibility_page,MONITOR
9,content_9532f197bbc8,0.166518,309192,104,high_visibility_page,MONITOR
10,content_36ff89c8214e,0.158927,295097,104,high_visibility_page,MONITOR
11,content_b28d1efd668f,0.154355,286608,104,high_visibility_page,MONITOR
19,content_813e88069237,0.125787,233561,104,high_visibility_page,MONITOR
1,content_2cb567c3c89b,0.123718,497727,48,high_visibility_page,MONITOR
23,content_c21024970297,0.113833,211366,104,high_visibility_page,MONITOR


### 1. content_5fe46e04994d

    Action: MONITOR
    Why: Very high visibility (517,715 impressions) and moderately stale (104 days).
    What would make it wrong:** The page may already be optimized and not require intervention.

### 2. content_2dba2b1f9536

    Action: MONITOR
    Why: High visibility and high baseline score driven by impressions and staleness.
    What would make it wrong: Traffic alone does not indicate content quality issues.

### 3. content_2c2606c5d176

    Action: MONITOR
    Why: Strong visibility (347,399 impressions) combined with moderate staleness (104 days).
    What would make it wrong: The page may continue to rank well without any content changes.

### 4. content_cb112fce36be
    
    Action: MONITOR
    Why: High visibility (309,910 impressions) and over three months since the last update.
    What would make it wrong: User intent and content quality may still be well aligned.

### 5. content_9532f197bbc8

    Action: MONITOR
    Why: Receives substantial traffic (309,192 impressions) and has not been updated for 104 days.
    What would make it wrong: The content could already be evergreen and not require refreshing.

### 6. content_36ff89c8214e

    Action: MONITOR
    Why: High visibility (295,097 impressions) and moderate staleness contribute to a strong baseline score.
    What would make it wrong: The page may not have any meaningful content gaps despite its age.

### 7. content_b28d1efd668f

    Action: MONITOR
    Why: Generates 286,608 impressions and has not been updated recently, making it worth monitoring.
    What would make it wrong: Traffic alone may be driving the ranking rather than actual refresh opportunity.

### 8. content_813e88069237

    Action: MONITOR
    Why: Strong search visibility (233,561 impressions) and 104 days since update.
    What would make it wrong: Performance metrics may remain healthy even without intervention.

### 9. content_2cb567c3c89b
    Action: MONITOR
    Why: Extremely high visibility (497,727 impressions). Despite being updated more recently (48 days), its traffic keeps it near the top.
    What would make it wrong: The page is relatively fresh and may not need review, suggesting the baseline still overweights visibility.

### 10. content_c21024970297
    Action: MONITOR
    Why: Over 211,000 impressions and 104 days since update indicate a potentially valuable page for review.
    What would make it wrong: High impressions do not guarantee that a refresh would improve outcomes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
top20["days_since_last_update"].value_counts()

,count
days_since_last_update,
104,19
48,1


In [ ]:
df["reason_code"].value_counts()
df["action"].value_counts()

,count
action,
MONITOR,29983
REFRESH,17


In [ ]:
df[df["action"] == "REFRESH"][
    [
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "baseline_score"
    ]
].head(20)

,content_id,impressions_90d,days_since_last_update,baseline_score
54,content_7f116ae1f6f5,954,301,0.001487
55,content_72496874f806,821,301,0.001280
83,content_cf56e2e2e282,61678,194,0.061963
85,content_7368877ea310,59472,194,0.059747
143,content_1bfaa38ff26c,25715,194,0.025834
228,content_0a91db491d14,13299,193,0.013292
239,content_5feee3994adb,7812,194,0.007848
245,content_c2d929d83eaa,7558,193,0.007554
254,content_b16bd7307b39,4590,194,0.004611
255,content_fe16a55cd13d,4556,194,0.004577


### Weak Pick Analysis

The baseline rule tends to favor pages with very high visibility and moderate staleness. Most top-ranked pages share the same update age (104 days), suggesting that the scoring function does not fully distinguish between different levels of refresh need.

One notable example is content_2cb567c3c89b, which ranks highly primarily because of its very high impressions despite being updated only 48 days ago. This indicates that the baseline may still overweight visibility relative to freshness.

Similarly, some REFRESH candidates have very low impression counts (<1000) but rank highly because they are extremely stale (183–301 days since update). A future ML model should learn a better balance between business impact and staleness.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.